# Host-Based Intrusion Detection & Automated Response System (HIDS)
**CT-378 Machine Learning - semester project.**

Goal: classify each host/connection record into one of five classes
(Normal, DoS, Probe, R2L, U2R) and map the predicted class to an automated
response action.

**Data:** NSL-KDD (Tavallaee et al., 2009). Two public files are combined:
`KDDTrain+.txt` and `KDDTest+.txt`. Source mirror:
https://github.com/defcom17/NSL_KDD . See `data/README.md`.

**Target:** `target` (5 classes). **Success metric:** macro-F1 (the data is
heavily imbalanced, so plain accuracy is misleading).

In [1]:
import sys, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib, sklearn

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from hids_core import *   # schema, add_features, response engine

warnings.filterwarnings("ignore")
RNG = 42
FIG = ROOT / "report" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
(ROOT / "models").mkdir(exist_ok=True)

# colour per class (categorical palette slots 1,2,3,7,8; fixed order)
CCOL = {"Normal": "#2a78d6", "DoS": "#eb6834", "Probe": "#1baf7a",
        "R2L": "#6250d6", "U2R": "#e34948"}
plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e5e5e2", "grid.linewidth": 0.6,
                     "axes.axisbelow": True})

## 1. Load and combine the two sources

In [2]:
a = load_raw(ROOT / "data" / "KDDTrain+.txt"); a["source"] = "KDDTrain+"
b = load_raw(ROOT / "data" / "KDDTest+.txt");  b["source"] = "KDDTest+"
raw = pd.concat([a, b], ignore_index=True)
print("rows before de-duplication:", len(raw))
dups = raw.duplicated(subset=FEATURES + ["target"]).sum()
df = raw.drop_duplicates(subset=FEATURES + ["target"]).reset_index(drop=True)
print("duplicates removed:", dups, "-> rows:", len(df), "| features:", len(FEATURES))
print("missing values:", int(df[FEATURES].isna().sum().sum()))
df["target"].value_counts()

rows before de-duplication: 148517
duplicates removed: 629 -> rows: 147888 | features: 41
missing values: 0


target
Normal    76967
DoS       52987
Probe     13935
R2L        3880
U2R         119
Name: count, dtype: int64

## 2. Exploratory data analysis (6 plots)

In [3]:
# Plot 1 - class balance
vc = df["target"].value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(4.2, 2.6))
ax.bar(vc.index, vc.values, color=[CCOL[c] for c in vc.index], width=0.6)
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=7)
ax.set_ylabel("records"); ax.set_title("Class balance (5 classes)")
fig.tight_layout(); fig.savefig(FIG / "eda_class_balance.png"); plt.show()
print((vc / vc.sum() * 100).round(2).to_dict())

{'Normal': 52.04, 'DoS': 35.83, 'Probe': 9.42, 'R2L': 2.62, 'U2R': 0.08}


In [4]:
# Plot 2 - protocol type by class (share within class)
ct = pd.crosstab(df["target"], df["protocol_type"], normalize="index").reindex(CLASSES) * 100
fig, ax = plt.subplots(figsize=(4.2, 2.6))
ct.plot(kind="bar", stacked=True, ax=ax, color=["#2a78d6", "#eda100", "#1baf7a"],
        width=0.6, edgecolor="white", linewidth=1)
ax.set_ylabel("% of class"); ax.set_xlabel(""); ax.set_title("Protocol by class")
ax.legend(title="", fontsize=7, frameon=False); plt.xticks(rotation=0)
fig.tight_layout(); fig.savefig(FIG / "eda_protocol.png"); plt.show()

In [5]:
# Plot 3 - outliers: log(src_bytes) per class (boxplot)
fig, ax = plt.subplots(figsize=(4.2, 2.6))
data = [np.log1p(df.loc[df.target == c, "src_bytes"]) for c in CLASSES]
bp = ax.boxplot(data, tick_labels=CLASSES, patch_artist=True, widths=0.5,
                flierprops=dict(marker=".", markersize=2, alpha=.3))
for p, c in zip(bp["boxes"], CLASSES):
    p.set_facecolor(CCOL[c]); p.set_alpha(.75)
ax.set_ylabel("log(1 + src_bytes)"); ax.set_title("Outliers: bytes sent per class")
fig.tight_layout(); fig.savefig(FIG / "eda_bytes_boxplot.png"); plt.show()

In [6]:
# Plot 4 - error rates by class (mean)
cols = ["serror_rate", "rerror_rate", "dst_host_serror_rate"]
m = df.groupby("target")[cols].mean().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(4.2, 2.6))
m.plot(kind="bar", ax=ax, color=["#2a78d6", "#eb6834", "#1baf7a"], width=0.75,
       edgecolor="white", linewidth=1)
ax.set_ylabel("mean rate"); ax.set_xlabel(""); ax.set_title("Connection error rates")
ax.legend(fontsize=6.5, frameon=False); plt.xticks(rotation=0)
fig.tight_layout(); fig.savefig(FIG / "eda_error_rates.png"); plt.show()

In [7]:
# Plot 5 - host-level privilege indicators by class (R2L / U2R signals)
cols = ["num_failed_logins", "root_shell", "su_attempted", "num_root", "num_shells", "num_file_creations"]
m = df.groupby("target")[cols].mean().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(4.2, 2.6))
im = ax.imshow(m.values.T, aspect="auto", cmap="Blues")
ax.set_xticks(range(5)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols, fontsize=7); ax.grid(False)
for i in range(5):
    for j in range(len(cols)):
        v = m.values[i, j]; ax.text(i, j, f"{v:.2f}", ha="center", va="center", fontsize=6,
                                      color="white" if v > m.values.max() * .55 else "black")
ax.set_title("Host-level indicators (class mean)")
fig.tight_layout(); fig.savefig(FIG / "eda_host_indicators.png"); plt.show()

In [8]:
# Plot 6 - correlation heatmap of numeric features
num = df[NUMERIC].loc[:, df[NUMERIC].std() > 0]
corr = num.corr()
fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1); ax.grid(False)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"Correlation of {corr.shape[0]} numeric features")
fig.colorbar(im, ax=ax, fraction=0.04)
fig.tight_layout(); fig.savefig(FIG / "eda_correlation.png"); plt.show()
hi = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
print("pairs with |r| > 0.9:", int((hi.abs() > 0.9).sum()))
print("constant columns dropped from the plot:", [c for c in NUMERIC if df[c].std() == 0])

pairs with |r| > 0.9: 13
constant columns dropped from the plot: ['num_outbound_cmds']


**EDA findings.** DoS and Normal dominate; U2R is extremely rare (<0.1%), so
macro-F1 and class weighting are needed. DoS shows very high SYN-error rates,
R2L shows failed logins / guest logins, U2R shows root-shell and `su` /
`num_root` signals. Many `*_rate` columns are strongly correlated.
`src_bytes` is heavy-tailed, hence the log transform in feature engineering.
There are no missing values (imputers are kept so the deployed pipeline is
robust to blank inputs).

## 3. Train / validation / test split (70 / 15 / 15, stratified)

In [9]:
from sklearn.model_selection import train_test_split
X, y = df[FEATURES], df["target"]
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RNG)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RNG)
print(len(X_tr), len(X_val), len(X_te))
pd.DataFrame({"train": y_tr.value_counts(), "val": y_val.value_counts(),
              "test": y_te.value_counts()}).reindex(CLASSES)

103521 22183 22184


,train,val,test
target,,,
Normal,53877,11545,11545
DoS,37091,7948,7948
Probe,9754,2090,2091
R2L,2716,582,582
U2R,83,18,18


## 4. Pre-processing pipeline (fitted on training data only)
Feature engineering (5 new columns) -> impute -> scale numerics / one-hot
categoricals -> classifier. Everything lives inside one scikit-learn `Pipeline`.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

def make_pipeline(clf):
    num_cols = NUMERIC + ENGINEERED
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         CATEGORICAL),
    ])
    return Pipeline([("fe", FunctionTransformer(add_features, validate=False)),
                     ("pre", pre), ("clf", clf)])

## 5. Baseline + three algorithms, tuned with 5-fold GridSearchCV
* Baseline: `DummyClassifier` (always predicts the majority class)
* Logistic Regression, Random Forest, Histogram Gradient Boosting
* Scoring: macro-F1; class weights are balanced to handle the imbalance.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score

cv = StratifiedKFold(5, shuffle=True, random_state=RNG)
specs = {
    "Logistic Regression": (LogisticRegression(max_iter=300, class_weight="balanced"),
                            {"clf__C": [0.1, 1, 10]}),
    "Random Forest": (RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample",
                                             n_jobs=2, random_state=RNG),
                      {"clf__max_depth": [15, None], "clf__min_samples_leaf": [1, 3]}),
    "Hist. Gradient Boosting": (HistGradientBoostingClassifier(max_iter=200, class_weight="balanced",
                                                               early_stopping=True, random_state=RNG),
                                {"clf__learning_rate": [0.05, 0.15], "clf__max_leaf_nodes": [15, 31]}),
}
rows, tuned = [], {}

base = make_pipeline(DummyClassifier(strategy="most_frequent")).fit(X_tr, y_tr)
rows.append({"Model": "Baseline (majority class)", "Best params": "-", "CV macro-F1": np.nan,
             "Train macro-F1": f1_score(y_tr, base.predict(X_tr), average="macro"),
             "Val macro-F1": f1_score(y_val, base.predict(X_val), average="macro"),
             "Val accuracy": accuracy_score(y_val, base.predict(X_val)), "Fit time (s)": 0})

for name, (clf, grid) in specs.items():
    t0 = time.time()
    gs = GridSearchCV(make_pipeline(clf), grid, cv=cv, scoring="f1_macro", n_jobs=1, refit=True)
    gs.fit(X_tr, y_tr)
    tuned[name] = gs.best_estimator_
    pv = gs.predict(X_val)
    rows.append({"Model": name, "Best params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()},
                 "CV macro-F1": gs.best_score_,
                 "Train macro-F1": f1_score(y_tr, gs.predict(X_tr), average="macro"),
                 "Val macro-F1": f1_score(y_val, pv, average="macro"),
                 "Val accuracy": accuracy_score(y_val, pv), "Fit time (s)": time.time() - t0})
    print(name, "done in", round(time.time() - t0), "s")

results = pd.DataFrame(rows)
results.round(4)

Logistic Regression done in 145 s
Random Forest done in 120 s
Hist. Gradient Boosting done in 161 s


,Model,Best params,CV macro-F1,Train macro-F1,Val macro-F1,Val accuracy,Fit time (s)
0,Baseline (majority class),-,NaN,0.1369,0.1369,0.5204,0.0000
1,Logistic Regression,{'C': 10},0.7432,0.7466,0.7411,0.9501,144.6916
2,Random Forest,"{'max_depth': None, 'min_samples_leaf': 3}",0.9355,0.9721,0.9415,0.9943,119.7402
3,Hist. Gradient Boosting,"{'learning_rate': 0.15, 'max_leaf_nodes': 31}",0.9188,0.9917,0.9402,0.9954,160.6869


Hist. Gradient Boosting done in 193 s


,Model,Best params,CV macro-F1,Train macro-F1,Val macro-F1,Val accuracy,Fit time (s)
0,Baseline (majority class),-,NaN,0.1369,0.1369,0.5204,0.0000
1,Logistic Regression,{'C': 10},0.7431,0.7478,0.7421,0.9505,127.4201
2,Random Forest,"{'max_depth': None, 'min_samples_leaf': 3}",0.9286,0.9701,0.9487,0.9943,63.0459
3,Hist. Gradient Boosting,"{'learning_rate': 0.15, 'max_leaf_nodes': 15}",0.9150,0.9612,0.9221,0.9923,192.5477


## 6. Model selection on the validation set

In [12]:
best_name = results.iloc[1:].sort_values("Val macro-F1", ascending=False).iloc[0]["Model"]
print("selected:", best_name)
results.to_csv(ROOT / "models" / "model_comparison.csv", index=False)
fig, ax = plt.subplots(figsize=(4.6, 2.6))
r = results.copy()
x = np.arange(len(r)); w = 0.38
ax.bar(x - w/2, r["Train macro-F1"], w, label="train", color="#2a78d6")
ax.bar(x + w/2, r["Val macro-F1"], w, label="validation", color="#eb6834")
ax.set_xticks(x); ax.set_xticklabels(["Baseline", "LogReg", "RandForest", "HistGB"], fontsize=7)
ax.set_ylabel("macro-F1"); ax.set_ylim(0, 1.05); ax.legend(frameon=False, fontsize=7)
ax.set_title("Train vs validation (over/under-fitting check)")
fig.tight_layout(); fig.savefig(FIG / "model_comparison.png"); plt.show()

selected: Random Forest


## 7. Final evaluation - held-out test set, used once

In [13]:
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             ConfusionMatrixDisplay)
final = tuned[best_name]
pred = final.predict(X_te)
proba = final.predict_proba(X_te)
test_macro = f1_score(y_te, pred, average="macro")
test_weighted = f1_score(y_te, pred, average="weighted")
test_acc = accuracy_score(y_te, pred)
test_auc = roc_auc_score(y_te, proba, multi_class="ovr", average="macro", labels=list(final.classes_))
print(f"TEST  accuracy={test_acc:.4f}  macro-F1={test_macro:.4f}  weighted-F1={test_weighted:.4f}  ROC-AUC(ovr)={test_auc:.4f}")
rep = classification_report(y_te, pred, labels=CLASSES, output_dict=True, zero_division=0)
print(classification_report(y_te, pred, labels=CLASSES, digits=4, zero_division=0))

cm = confusion_matrix(y_te, pred, labels=CLASSES)
fig, ax = plt.subplots(figsize=(4.0, 3.4))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.grid(False); ax.set_title(f"Test confusion matrix - {best_name}")
fig.tight_layout(); fig.savefig(FIG / "confusion_matrix.png"); plt.show()

TEST  accuracy=0.9947  macro-F1=0.9125  weighted-F1=0.9947  ROC-AUC(ovr)=0.9963
              precision    recall  f1-score   support

      Normal     0.9981    0.9922    0.9951     11545
         DoS     0.9987    0.9996    0.9992      7948
       Probe     0.9877    0.9986    0.9931      2091
         R2L     0.9159    0.9725    0.9433       582
         U2R     0.6000    0.6667    0.6316        18

    accuracy                         0.9947     22184
   macro avg     0.9001    0.9259    0.9125     22184
weighted avg     0.9949    0.9947    0.9947     22184



              precision    recall  f1-score   support

      Normal     0.9979    0.9927    0.9953     11545
         DoS     0.9987    0.9995    0.9991      7948
       Probe     0.9868    0.9986    0.9926      2091
         R2L     0.9264    0.9725    0.9489       582
         U2R     0.6667    0.6667    0.6667        18

    accuracy                         0.9949     22184
   macro avg     0.9153    0.9260    0.9205     22184
weighted avg     0.9950    0.9949    0.9949     22184



### Error analysis

In [14]:
err = pd.DataFrame({"true": y_te.values, "pred": pred})
err = err[err.true != err.pred].value_counts().reset_index(name="n").head(8)
print("total test errors:", int((y_te.values != pred).sum()), "of", len(y_te))
err

total test errors: 118 of 22184


,true,pred,n
0,Normal,R2L,52
1,Normal,Probe,25
2,R2L,Normal,12
3,Normal,DoS,9
4,U2R,Normal,6
5,R2L,U2R,4
6,Normal,U2R,4
7,Probe,Normal,2


## 8. Which features matter? (permutation importance on validation data)

In [15]:
from sklearn.inspection import permutation_importance
sub = X_val.sample(5000, random_state=RNG); ysub = y_val.loc[sub.index]
pi = permutation_importance(final, sub, ysub, scoring="f1_macro", n_repeats=3,
                            random_state=RNG, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
imp.rename("importance").to_csv(ROOT / "models" / "feature_importance.csv")
top = imp.head(12)[::-1]
fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.barh(top.index, top.values, color="#2a78d6", height=0.6)
ax.set_xlabel("drop in macro-F1 when shuffled"); ax.set_title("Top-12 features (permutation)")
fig.tight_layout(); fig.savefig(FIG / "feature_importance.png"); plt.show()
imp.head(12)

src_bytes                 0.087683
dst_bytes                 0.060408
num_file_creations        0.056642
service                   0.038771
duration                  0.015100
su_attempted              0.012083
dst_host_srv_count        0.009253
dst_host_count            0.005030
protocol_type             0.004871
dst_host_same_srv_rate    0.004764
same_srv_rate             0.001412
srv_rerror_rate           0.001148
dtype: float64

## 9. Save the final pipeline (joblib) and metadata for the apps

In [16]:
joblib.dump(final, ROOT / "models" / "hids_pipeline.joblib", compress=3)

top_features = list(imp.head(12).index)
defaults = {}
for c in FEATURES:
    defaults[c] = (X_tr[c].mode().iloc[0] if c in CATEGORICAL else float(X_tr[c].median()))
ranges = {c: [float(X_tr[c].min()), float(X_tr[c].max())] for c in NUMERIC}

# demo events: 4 correctly-classified test records per class (kept in the repo for the apps)
ev = X_te.copy(); ev["true_class"] = y_te.values; ev["pred"] = pred
ev = ev[ev.true_class == ev.pred]
ev = pd.concat([ev[ev.true_class == c].sample(4, random_state=RNG) for c in CLASSES])
ev = ev.drop(columns="pred").reset_index(drop=True)
ev.to_csv(ROOT / "app" / "sample_events.csv", index=False)

meta = {
    "classes": list(final.classes_), "best_model": best_name,
    "best_params": results.set_index("Model").loc[best_name, "Best params"],
    "test": {"accuracy": test_acc, "macro_f1": test_macro, "weighted_f1": test_weighted,
             "roc_auc_ovr_macro": test_auc,
             "per_class_f1": {c: rep[c]["f1-score"] for c in CLASSES}},
    "top_features": top_features, "defaults": defaults, "ranges": ranges,
    "services": sorted(df["service"].unique()), "protocols": sorted(df["protocol_type"].unique()),
    "flags": sorted(df["flag"].unique()),
    "n_rows": int(len(df)), "sklearn_version": sklearn.__version__,
}
json.dump(meta, open(ROOT / "models" / "metadata.json", "w"), indent=2, default=str)

# sanity check: reload and re-predict
reloaded = joblib.load(ROOT / "models" / "hids_pipeline.joblib")
assert (reloaded.predict(X_te.head(200)) == final.predict(X_te.head(200))).all()
print("saved. sklearn", sklearn.__version__)

saved. sklearn 1.6.1


# Experiment 2: Autoencoder-Based Anomaly Detection

Experiment 1 (the Random Forest 5-class classifier, its splits, preprocessing, saved model
and deployments) is **left completely untouched**. This is a separate experiment appended at
the end of the notebook.

**Idea.** An autoencoder (AE) is trained to reconstruct *Normal* traffic. Attack records
should reconstruct worse, so the reconstruction error acts as an unsupervised anomaly score.
Records flagged as anomalous are then passed to the **already-trained Random Forest** from
Experiment 1, which picks the attack category -> a **two-stage hybrid detector**.

Why it can help: the AE never sees attack labels, so it can flag unusual traffic on its own.
Why it can hurt: attacks whose error stays below the threshold are missed by Stage 1 and can
never be recovered by Stage 2.

**No-leakage rules used in this section:**

| Data | Used for |
|---|---|
| Normal rows of `X_tr` | fitting the AE preprocessor **and** training the AE |
| Normal rows of `X_val` | monitoring training and choosing the threshold **only** |
| `X_te` (all five classes) | final evaluation **only**, at the very end |

Everything below reuses the existing `X_tr/X_val/X_te`, `y_tr/y_val/y_te`, `make_pipeline`
and `final` (Random Forest) objects without modifying them.

## E2.1 Configuration

All knobs for this experiment live in one place. Changing them does not affect Experiment 1.

In [17]:
# --- Experiment 2 settings (safe to change; Experiment 1 is unaffected) ---
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # quieter TensorFlow import

AE_PERCENTILE = 95          # threshold = this percentile of NORMAL validation errors
AE_HIDDEN = (64, 32, 16)    # encoder 64 -> 32 -> bottleneck 16 (decoder mirrors it)
AE_EPOCHS = 60              # maximum epochs (early stopping usually stops sooner)
AE_BATCH = 256
AE_LR = 1e-3
AE_PATIENCE = 8             # epochs without validation improvement before stopping

print("autoencoder hidden layers:", AE_HIDDEN, "| threshold percentile:", AE_PERCENTILE)

autoencoder hidden layers: (64, 32, 16) | threshold percentile: 95


## E2.2 Prepare the autoencoder data (Normal only, no validation/test fitting)

* Only the **Normal** rows of the original training split are used.
* The preprocessing is the **same definition** as Experiment 1: the notebook's
  `make_pipeline` is built with the classifier replaced by `"passthrough"`, so it reuses
  `add_features`, the median/most-frequent imputers, the `StandardScaler` and the one-hot
  encoder without duplicating preprocessing code.
* A **new preprocessor instance** is fitted on Normal training rows only. Attack rows,
  validation rows and test rows never influence the imputer/scaler/one-hot statistics.
* The AE receives a pure numeric matrix: scaled numerics + 5 engineered features + one-hot
  categoricals (Keras needs numbers, not strings).

In [18]:
# 1) Normal-only subsets of the original splits
X_tr_norm = X_tr[y_tr == "Normal"]
X_val_norm = X_val[y_val == "Normal"]
print(f"normal training rows: {len(X_tr_norm):,} | normal validation rows: {len(X_val_norm):,}")

# 2) Reuse the existing preprocessing definition with the classifier disabled.
#    Fitting only on Normal training rows keeps attack rows out of the fit.
ae_pre = make_pipeline("passthrough")
ae_pre.fit(X_tr_norm)

# Keras works on float32 arrays
def to_ae(X):
    return np.asarray(ae_pre.transform(X), dtype="float32")

Xa_tr = to_ae(X_tr_norm)    # AE input = AE target (reconstruction)
Xa_val = to_ae(X_val_norm)  # Normal validation: monitoring + threshold only
Xa_te = to_ae(X_te)         # untouched test set: used for evaluation only
print("processed feature count:", Xa_tr.shape[1])
print("shapes -> train:", Xa_tr.shape, "| val:", Xa_val.shape, "| test:", Xa_te.shape)

normal training rows: 53,877 | normal validation rows: 11,545
processed feature count: 83
shapes -> train: (53877, 83) | val: (11545, 83) | test: (22184, 83)


## E2.3 Autoencoder architecture

Input (processed features) -> 64 (ReLU) -> 32 (ReLU) -> **16 (bottleneck, ReLU)** ->
32 (ReLU) -> 64 (ReLU) -> output (same width as the input, **linear** activation).

The narrow bottleneck forces the network to keep only the dominant structure of Normal
traffic. Loss: mean squared error (MSE) between input and reconstruction; optimiser: Adam.

In [21]:
!pip install tensorflow
import tensorflow as tf
tf.keras.utils.set_random_seed(RNG)   # reproducible weight initialisation

n_features = Xa_tr.shape[1]
inp = tf.keras.Input(shape=(n_features,), name="input")
x = inp
for i, units in enumerate(AE_HIDDEN):                 # encoder: 64 -> 32 -> 16 (bottleneck)
    x = tf.keras.layers.Dense(units, activation="relu", name=f"enc_{i}")(x)
for i, units in enumerate(reversed(AE_HIDDEN[:-1])):  # decoder: 32 -> 64
    x = tf.keras.layers.Dense(units, activation="relu", name=f"dec_{i}")(x)
out = tf.keras.layers.Dense(n_features, activation="linear", name="reconstruction")(x)

autoencoder = tf.keras.Model(inp, out, name="autoencoder")
autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=AE_LR), loss="mse")
autoencoder.summary()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 1.3 MB/s eta 0:00:0000:0100:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 1.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 2.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 1.6 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: protobuf━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/15 [libclang]
    Found existing installation: protobuf 5.29.3━━━━━━━━━━━━━━━━━━  4/15 [protobuf]
    Uninstalling protobuf-5.29.3:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/15 [protobuf]
      Successfully uninstalled protobuf-5.29.3━━━━━━━━━━━━━━━━  4/15 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [tensorflow]5 [tensorflow]a]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires protobuf<7,>=3.20, but you have protobuf 7

I0000 00:00:1795547355.614430     717 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795547358.992414     717 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1795547359.568841     717 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 83)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_0 (Dense)                   │ (None, 64)             │         5,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_1 (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_2 (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_0 (Dense)                   │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_1 (Dense)                   │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction (Dense)          │ (None, 83)             │         5,395 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,035 (62.64 KB)

 Trainable params: 16,035 (62.64 KB)

 Non-trainable params: 0 (0.00 B)

## E2.4 Train on Normal traffic only

Every training row is its own target. Early stopping monitors the MSE on the held-out
**Normal validation** rows and restores the best weights, so neither attack samples nor the
test set influence training.

In [22]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=AE_PATIENCE,
                                              restore_best_weights=True)]
history = autoencoder.fit(
    Xa_tr, Xa_tr,                       # autoencoder: reconstruct its own input
    validation_data=(Xa_val, Xa_val),   # Normal validation rows, monitoring only
    epochs=AE_EPOCHS, batch_size=AE_BATCH, shuffle=True, verbose=0,
    callbacks=callbacks,
)
print(f"epochs run: {len(history.history['loss'])} | best normal-validation MSE: "
      f"{min(history.history['val_loss']):.6f}")

fig, ax = plt.subplots(figsize=(4.6, 2.6))
ax.plot(history.history["loss"], label="train (Normal)", color="#2a78d6")
ax.plot(history.history["val_loss"], label="validation (Normal)", color="#eb6834")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss")
ax.set_title("Autoencoder training"); ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "ae_training_curve.png"); plt.show()

epochs run: 60 | best normal-validation MSE: 0.016533


## E2.5 Reconstruction error and threshold (Normal validation only)

The per-record error is the mean squared reconstruction error over all processed features.
The threshold is the **`AE_PERCENTILE`-th percentile** of the errors of the Normal
**validation** rows. At the default 95, roughly 5% of normal traffic becomes a false alarm
by construction; a stricter value (e.g. 99) lowers false alarms but also lowers attack
recall. The test set is not touched in this step.

In [23]:
def reconstruction_mse(Xa):
    recon = autoencoder.predict(Xa, batch_size=4096, verbose=0)
    return np.mean((Xa - recon) ** 2, axis=1)

err_tr = reconstruction_mse(Xa_tr)    # Normal training (reference only)
err_val = reconstruction_mse(Xa_val)  # Normal validation -> threshold
err_te = reconstruction_mse(Xa_te)    # test -> final evaluation only (below)

THRESHOLD = float(np.percentile(err_val, AE_PERCENTILE))
print(f"threshold ({AE_PERCENTILE}th pct of normal-validation MSE): {THRESHOLD:.6f}")
print("normal-validation MSE quantiles:",
      {f"p{q}": round(float(np.percentile(err_val, q)), 6) for q in (50, 90, 95, 99, 100)})

fig, ax = plt.subplots(figsize=(4.6, 2.6))
ax.hist(err_val, bins=80, color="#2a78d6", alpha=.85)
ax.axvline(THRESHOLD, color="#e34948", ls="--", lw=1.2, label=f"threshold = {THRESHOLD:.5f}")
ax.set_yscale("log")
ax.set_xlabel("reconstruction MSE (Normal validation)"); ax.set_ylabel("records")
ax.set_title("Normal reconstruction errors and the chosen threshold")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "ae_threshold.png"); plt.show()

threshold (95th pct of normal-validation MSE): 0.040387
normal-validation MSE quantiles: {'p50': 0.002634, 'p90': 0.02255, 'p95': 0.040387, 'p99': 0.159953, 'p100': 23.930399}


## E2.6 Stage 1: binary detection on the untouched test set

Decision rule: `error <= threshold` -> **Normal**, `error > threshold` -> **Attack**.

The classes are very imbalanced, so besides overall accuracy we report precision, recall and
F1 for the **Attack** class, the false-positive rate (normal traffic wrongly flagged), and
the detection rate per true class (what fraction of each class Stage 1 flags).

In [24]:
from sklearn.metrics import precision_score, recall_score

y_te_bin = (y_te.values != "Normal").astype(int)   # 1 = attack, 0 = normal
pred_bin = (err_te > THRESHOLD).astype(int)        # Stage-1 decision

tn, fp, fn, tp = confusion_matrix(y_te_bin, pred_bin, labels=[0, 1]).ravel()
print(f"binary accuracy  : {accuracy_score(y_te_bin, pred_bin):.4f}")
print(f"attack precision : {precision_score(y_te_bin, pred_bin):.4f}")
print(f"attack recall    : {recall_score(y_te_bin, pred_bin):.4f}")
print(f"attack F1        : {f1_score(y_te_bin, pred_bin):.4f}")
print(f"false-positive rate: {fp / (fp + tn):.4f}  (FP={fp:,} of {fp + tn:,} normal rows)")
print(f"missed attacks   : {fn:,} of {fn + tp:,}")

fig, ax = plt.subplots(figsize=(3.4, 2.8))
ConfusionMatrixDisplay(np.array([[tn, fp], [fn, tp]]),
                       display_labels=["Normal", "Attack"]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d")
ax.grid(False); ax.set_title("Stage 1 - test confusion matrix")
fig.tight_layout(); fig.savefig(FIG / "ae_binary_confusion.png"); plt.show()

# what fraction of each TRUE class is flagged as anomalous
det = pd.DataFrame({"true_class": y_te.values, "flagged": pred_bin})
det_rate = det.groupby("true_class")["flagged"].agg(["sum", "count", "mean"]).reindex(CLASSES)
det_rate.columns = ["flagged", "support", "detection rate (%)"]
det_rate["detection rate (%)"] = (det_rate["detection rate (%)"] * 100).round(2)
det_rate

binary accuracy  : 0.9436
attack precision : 0.9432
attack recall    : 0.9388
attack F1        : 0.9410
false-positive rate: 0.0521  (FP=601 of 11,545 normal rows)
missed attacks   : 651 of 10,639


,flagged,support,detection rate (%)
true_class,,,
Normal,601,11545,5.21
DoS,7883,7948,99.18
Probe,1892,2091,90.48
R2L,200,582,34.36
U2R,13,18,72.22


## E2.7 Reconstruction error per true class

How well does the AE separate the five classes on its own? Below: mean, median, spread and
5th/95th percentiles of the test reconstruction error per class. U2R has only 18 test
records, so read that row with care.

In [25]:
err_tbl = pd.DataFrame({"true_class": y_te.values, "mse": err_te, "flagged": pred_bin})
stats = err_tbl.groupby("true_class")["mse"].agg(["count", "mean", "median", "std"])
q = err_tbl.groupby("true_class")["mse"].quantile([0.05, 0.95]).unstack()
stats = stats.join(q).rename(columns={0.05: "p05", 0.95: "p95"}).reindex(CLASSES)
stats["% flagged"] = (err_tbl.groupby("true_class")["flagged"].mean() * 100).round(2)
stats.round(6)

,count,mean,median,std,p05,p95,% flagged
true_class,,,,,,,
Normal,11545,0.015800,0.002548,0.123231,0.000298,0.041968,5.21
DoS,7948,0.723716,0.809097,0.396282,0.086274,1.268095,99.18
Probe,2091,0.642327,0.080910,12.899409,0.031272,1.437140,90.48
R2L,582,0.223221,0.030097,1.460703,0.010213,0.736587,34.36
U2R,18,3.093959,0.119784,10.620904,0.010552,11.692670,72.22


Visualising the same errors: (left) boxplot per class on a log scale, (right) overlaid
distributions of `log10(MSE)`. The dashed line is the threshold - classes whose mass sits to
the right of it are detected well.

In [26]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.2))

# (a) per-class error distribution
bp = axes[0].boxplot([err_te[y_te.values == c] for c in CLASSES], tick_labels=CLASSES,
                     patch_artist=True, widths=.55,
                     flierprops=dict(marker=".", markersize=2, alpha=.35))
for patch, c in zip(bp["boxes"], CLASSES):
    patch.set_facecolor(CCOL[c]); patch.set_alpha(.8)
axes[0].axhline(THRESHOLD, color="#e34948", ls="--", lw=1.1)
axes[0].set_yscale("log"); axes[0].set_ylabel("reconstruction MSE")
axes[0].set_title("Reconstruction error by true class")

# (b) distributions of log10(MSE) per class
small = np.maximum(err_te, 1e-8)
for c in CLASSES:
    axes[1].hist(np.log10(small[y_te.values == c]), bins=60, histtype="step", lw=1.4,
                 density=True, color=CCOL[c], label=c)
axes[1].axvline(np.log10(THRESHOLD), color="#e34948", ls="--", lw=1.1)
axes[1].set_xlabel("log10(reconstruction MSE)"); axes[1].set_ylabel("density")
axes[1].set_title("Error distributions (dashed = threshold)")
axes[1].legend(frameon=False, fontsize=6.5)

fig.tight_layout(); fig.savefig(FIG / "ae_reconstruction_errors.png"); plt.show()

## E2.8 Stage 2 and the complete two-stage system

Records flagged by Stage 1 are passed **as original raw rows** (the 41 input columns) to the
Experiment 1 Random Forest pipeline `final`, which applies its own preprocessing. The AE's
reconstructed output is never given to it.

Final label = `Normal` when Stage 1 says normal, otherwise the Random Forest category
(which may itself be `Normal`).

An attack missed by Stage 1 becomes a `Normal` prediction end-to-end and is counted as a
system error below - Stage 2 never gets the chance to classify it.

In [27]:
# Stage 2 receives the ORIGINAL records flagged by Stage 1 (not AE reconstructions)
flagged_mask = pred_bin == 1
X_te_flagged = X_te.loc[flagged_mask]
print(f"stage 1 flagged {len(X_te_flagged):,} of {len(X_te):,} test records")

cat_pred = final.predict(X_te_flagged)          # Experiment-1 Random Forest, unchanged
print("Random Forest categories on flagged records:")
print(pd.Series(cat_pred).value_counts().to_string())

# per-class quality of Stage 2 on the flagged subset (ground truth known for the experiment)
print()
print("Stage 2 classification report (flagged records only):")
print(classification_report(y_te.loc[flagged_mask], cat_pred, labels=CLASSES,
                            digits=4, zero_division=0))

# end-to-end prediction for every test record
hybrid_pred = np.full(len(X_te), "Normal", dtype=object)
hybrid_pred[flagged_mask] = cat_pred
print("records the Random Forest re-labelled as Normal:", int((cat_pred == "Normal").sum()))

stage 1 flagged 10,589 of 22,184 test records
Random Forest categories on flagged records:
DoS       7892
Probe     1909
Normal     572
R2L        196
U2R         20

Stage 2 classification report (flagged records only):
              precision    recall  f1-score   support

      Normal     0.9913    0.9434    0.9668       601
         DoS     0.9987    0.9999    0.9993      7883
       Probe     0.9900    0.9989    0.9945      1892
         R2L     0.9847    0.9650    0.9747       200
         U2R     0.6000    0.9231    0.7273        13

    accuracy                         0.9958     10589
   macro avg     0.9129    0.9661    0.9325     10589
weighted avg     0.9960    0.9958    0.9958     10589

records the Random Forest re-labelled as Normal: 572


### End-to-end evaluation (includes Stage-1 misses)

This is the honest system score: Stage-1 misses are `true attack -> Normal` errors here, and
false alarms cost precision. Because of the imbalance, read macro-F1 and the per-class
support together with accuracy.

In [ ]:
e2e_acc = accuracy_score(y_te, hybrid_pred)
e2e_macro = f1_score(y_te, hybrid_pred, average="macro")
e2e_weighted = f1_score(y_te, hybrid_pred, average="weighted")
print(f"END-TO-END  accuracy={e2e_acc:.4f}  macro-F1={e2e_macro:.4f}  "
      f"weighted-F1={e2e_weighted:.4f}")
print(classification_report(y_te, hybrid_pred, labels=CLASSES, digits=4, zero_division=0))

fig, ax = plt.subplots(figsize=(4.0, 3.4))
ConfusionMatrixDisplay(confusion_matrix(y_te, hybrid_pred, labels=CLASSES),
                       display_labels=CLASSES).plot(ax=ax, cmap="Blues",
                                                    colorbar=False, values_format="d")
ax.grid(False); ax.set_title("End-to-end confusion matrix (AE + RF)")
fig.tight_layout(); fig.savefig(FIG / "ae_hybrid_confusion.png"); plt.show()

# where do end-to-end errors come from?
e2e_err = pd.DataFrame({"true": y_te.values, "pred": hybrid_pred,
                        "stage1_flagged": flagged_mask})
e2e_err = e2e_err[e2e_err.true != e2e_err.pred]
print("end-to-end errors:", len(e2e_err), "of", len(y_te),
      "| missed by stage 1:", int((~e2e_err.stage1_flagged).sum()))
e2e_err.groupby(["true", "pred", "stage1_flagged"]).size().reset_index(name="n").head(10)

## E2.9 Experiment 1 vs Experiment 2 on the same test records

Same `X_te` / `y_te`, and no re-training: `pred_rf` is just the Experiment 1 pipeline again.
The binary rows (attack precision/recall, false-positive rate) collapse the five classes into
normal vs attack and make the cost of the Stage-1 gate visible.

In [ ]:
def evaluate_system(y_true, y_pred):
    yb = (np.asarray(y_true) != "Normal").astype(int)
    pb = (np.asarray(y_pred) != "Normal").astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(yb, pb, labels=[0, 1]).ravel()
    f1s = f1_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Macro-F1": f1_score(y_true, y_pred, average="macro"),
            "Weighted-F1": f1_score(y_true, y_pred, average="weighted"),
            "Attack precision": precision_score(yb, pb, zero_division=0),
            "Attack recall": recall_score(yb, pb),
            "False-positive rate": fp_ / (fp_ + tn_),
            **{f"F1 {c}": v for c, v in zip(CLASSES, f1s)}}

pred_rf = final.predict(X_te)   # Experiment-1 output (same model, no re-training)
comparison = pd.DataFrame({"Random Forest (Exp. 1)": evaluate_system(y_te, pred_rf),
                           "AE + RF hybrid (Exp. 2)": evaluate_system(y_te, hybrid_pred)})
comparison.round(4)

Per-class F1 side by side (the hybrid can only lose where Stage 1 misses an attack).

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 2.8))
xpos = np.arange(len(CLASSES)); w = .38
ax.bar(xpos - w/2, [comparison.loc[f"F1 {c}", "Random Forest (Exp. 1)"] for c in CLASSES],
       w, label="Random Forest (Exp. 1)", color="#2a78d6")
ax.bar(xpos + w/2, [comparison.loc[f"F1 {c}", "AE + RF hybrid (Exp. 2)"] for c in CLASSES],
       w, label="AE + RF hybrid (Exp. 2)", color="#eb6834")
ax.set_xticks(xpos); ax.set_xticklabels(CLASSES); ax.set_ylim(0, 1.05)
ax.set_ylabel("F1-score"); ax.set_title("Per-class F1 on the same test set")
ax.legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "ae_vs_rf_comparison.png"); plt.show()

## E2.10 What the experiment shows

* **No leakage.** The preprocessor and the AE saw only Normal training rows; the threshold
  used only Normal validation rows; the test set was evaluated once, at the end.
* **Stage 1 caps the hybrid.** Any attack with `error <= threshold` becomes a `Normal`
  prediction end-to-end. The drop in attack recall and per-class F1 versus Experiment 1 is
  exactly the price of the unsupervised gate.
* **Class imbalance.** Macro-F1 and the per-class table (with support) are the fair metrics;
  U2R has only 18 test rows, so its numbers are noisy.
* **One knob.** `AE_PERCENTILE` controls the normal/attack trade-off: 95 flags ~5% of normal
  traffic, 99 is stricter but misses more attacks.
* **Preservation.** Experiment 1's cells, hyperparameters, split, saved model
  (`models/hids_pipeline.joblib`) and deployments are unchanged; this section adds no model
  files (only new figures in `report/figures/`).